# 07 — All Remaining Clean-Room Models in One Run (#10 Real-PLKSR, #13 Clean MI-GAN, #4 Pure-COCO RTMPose-S)
Trains all three remaining models on one T4 GPU (~20-25 min total) and saves all 3 ONNX files straight to `/content/drive/MyDrive/infinite_ml/`:
1. **`real_plksr_clean.onnx`** (#10, MIT, 1x Restore/Denoise/DeJPEG on PD12M CC0 + `e1_200` + LPIPS BSD-2)
2. **`migan_clean_512.onnx`** (#13, MIT, Places2-free 512x512 Inpaint on PD12M CC0 + `e1_200` + LPIPS BSD-2)
3. **`rtmpose_s_coco_clean.onnx`** (#4, Apache-2.0, Pure-COCO Keypoints without AI Challenger / MPII taint)


In [ ]:
!pip -q uninstall -y torchaudio
!pip -q install -U torch torchvision timm onnx onnxscript onnxslim onnxruntime lpips datasets pillow numpy tqdm
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/out', exist_ok=True)
!unzip -q -o /content/drive/MyDrive/infinite_ml/e1_200.zip -d /content/


In [ ]:
# Shared clean data: stream 300 CC0 photos from Spawning/PD12M
import os, glob, random, shutil, hashlib, urllib.request, json
from io import BytesIO
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, onnx, timm, lpips
from PIL import Image, ImageDraw, ImageFilter
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

pd_dir = '/content/pd12m_300'
os.makedirs(pd_dir, exist_ok=True)
if len(glob.glob(f'{pd_dir}/*.jpg')) < 260:
    ds_stream = load_dataset('Spawning/PD12M', split='train', streaming=True)
    saved = 0
    for row in ds_stream:
        if saved >= 300: break
        url = row.get('url') or row.get('image_url') or ''
        if not url.startswith('http'): continue
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'infinite-ml'})
            with urllib.request.urlopen(req, timeout=5) as r:
                im = Image.open(BytesIO(r.read())).convert('RGB')
            if min(im.size) < 256: continue
            im.resize((512, 512), Image.BILINEAR).save(f'{pd_dir}/pd_{saved:04d}.jpg', quality=92)
            saved += 1
        except Exception:
            continue
    print('Fetched PD12M CC0 photos:', saved)
clean_imgs = sorted(glob.glob(f'{pd_dir}/*.jpg')) + sorted(glob.glob('/content/e1_200/scene_*/composite.png'))
lpips_fn = lpips.LPIPS(net='alex').cuda()
for p in lpips_fn.parameters(): p.requires_grad = False
print('Clean corpus ready:', len(clean_imgs))


In [ ]:
# ==================== 1/3: #10 CLEAN REAL-PLKSR (1x Heavy Restore/DeJPEG/Denoise) ====================
class PLKBlock(nn.Module):
    def __init__(self, c=48):
        super().__init__()
        self.pw1 = nn.Conv2d(c, c, 1)
        self.dw  = nn.Conv2d(c // 2, c // 2, 13, padding=6, groups=c // 2) # partial large kernel (EP-safe)
        self.pw2 = nn.Conv2d(c, c, 1)
        self.act = nn.Mish(inplace=True)
    def forward(self, x):
        h = self.act(self.pw1(x))
        h1, h2 = torch.chunk(h, 2, dim=1)
        h = torch.cat([self.dw(h1), h2], dim=1)
        return x + self.pw2(h)

class CleanRealPLKSR(nn.Module):
    def __init__(self, c=48, blocks=6):
        super().__init__()
        self.head = nn.Conv2d(3, c, 3, padding=1)
        self.body = nn.Sequential(*[PLKBlock(c) for _ in range(blocks)])
        self.tail = nn.Conv2d(c, 3, 3, padding=1)
    def forward(self, x):
        return torch.clamp(x + self.tail(self.body(self.head(x))), 0.0, 1.0)

class RestoreDS(Dataset):
    def __init__(self, paths, sz=256):
        self.paths, self.sz = paths, sz
    def __len__(self): return len(self.paths) * 2
    def __getitem__(self, i):
        im = Image.open(self.paths[i % len(self.paths)]).convert('RGB')
        w, h = im.size
        x0, y0 = random.randint(0, max(0, w - self.sz)), random.randint(0, max(0, h - self.sz))
        hr = im.crop((x0, y0, x0 + self.sz, y0 + self.sz)).resize((self.sz, self.sz), Image.BICUBIC)
        deg = hr
        if random.random() < 0.7:
            deg = deg.filter(ImageFilter.GaussianBlur(random.uniform(0.4, 1.4)))
        if random.random() < 0.85:
            buf = BytesIO(); deg.save(buf, format='JPEG', quality=random.randint(25, 70)); buf.seek(0)
            deg = Image.open(buf).convert('RGB')
        t_hr = torch.from_numpy(np.asarray(hr, np.float32) / 255.0).permute(2,0,1)
        t_lr = torch.from_numpy(np.asarray(deg, np.float32) / 255.0).permute(2,0,1)
        if random.random() < 0.5:
            t_lr = torch.clamp(t_lr + torch.randn_like(t_lr) * random.uniform(0.01, 0.04), 0.0, 1.0)
        return t_lr, t_hr

dl_r = DataLoader(RestoreDS(clean_imgs, 256), batch_size=16, shuffle=True, num_workers=2)
plksr = CleanRealPLKSR(c=48, blocks=6).cuda()
opt = torch.optim.AdamW(plksr.parameters(), lr=6e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=12)
for ep in range(12):
    tot = 0.0
    for lr, hr in dl_r:
        lr, hr = lr.cuda(), hr.cuda()
        pred = plksr(lr)
        loss = F.l1_loss(pred, hr) + 0.04 * lpips_fn(pred * 2 - 1, hr * 2 - 1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()
    sched.step()
    if (ep + 1) in (1, 6, 12): print(f"Real-PLKSR Epoch {ep+1}/12 loss={tot/len(dl_r):.4f}")

p_plk = '/content/out/real_plksr_clean.onnx'
torch.onnx.export(plksr.eval().cpu(), torch.randn(1,3,256,256), p_plk, opset_version=17, dynamo=False, input_names=['input'], output_names=['output'])
onnx.checker.check_model(p_plk)
d = open(p_plk, 'rb').read()
shutil.copy2(p_plk, '/content/drive/MyDrive/infinite_ml/real_plksr_clean.onnx')
print(f"Real-PLKSR Exported: {len(d)/1e6:.2f} MB sha256={hashlib.sha256(d).hexdigest()}")
del plksr, dl_r; torch.cuda.empty_cache()


In [ ]:
# ==================== 2/3: #13 CLEAN MI-GAN 512 INPAINT (Zero Places2, Zero FFT) ====================
class CleanMigan512(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(4, 32, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(inplace=True))
        self.enc2 = nn.Sequential(nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(inplace=True), nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(inplace=True))
        self.enc3 = nn.Sequential(nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(inplace=True), nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(inplace=True))
        self.mid  = nn.Sequential(
            nn.Conv2d(128, 128, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=4, dilation=4), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=8, dilation=8), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(inplace=True),
        )
        self.up2  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec2 = nn.Sequential(nn.Conv2d(128+64, 64, 3, padding=1), nn.ReLU(inplace=True))
        self.up1  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec1 = nn.Sequential(nn.Conv2d(64+32, 32, 3, padding=1), nn.ReLU(inplace=True))
        self.out  = nn.Sequential(nn.Conv2d(32, 3, 3, padding=1), nn.Sigmoid())
    def forward(self, image, mask):
        # image: [1,3,512,512] float in [0,1], mask: [1,1,512,512] float (1.0 = hole)
        x = torch.cat([image * (1.0 - mask), mask], dim=1)
        e1 = self.enc1(x); e2 = self.enc2(e1); e3 = self.enc3(e2)
        m = self.mid(e3)
        d2 = self.dec2(torch.cat([self.up2(m), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        pred = self.out(d1)
        return image * (1.0 - mask) + pred * mask

class InpaintDS(Dataset):
    def __init__(self, paths, sz=512):
        self.paths, self.sz = paths, sz
    def __len__(self): return len(self.paths) * 2
    def __getitem__(self, i):
        im = Image.open(self.paths[i % len(self.paths)]).convert('RGB').resize((self.sz, self.sz), Image.BILINEAR)
        m = Image.new('L', (self.sz, self.sz), 0)
        dr = ImageDraw.Draw(m)
        for _ in range(random.randint(2, 5)):
            if random.random() < 0.5:
                x0, y0 = random.randint(20, 380), random.randint(20, 380)
                dr.rectangle([x0, y0, x0 + random.randint(30, 120), y0 + random.randint(30, 120)], fill=255)
            else:
                pts = [(random.randint(30, 480), random.randint(30, 480)) for _ in range(random.randint(3, 6))]
                dr.line(pts, fill=255, width=random.randint(18, 46))
        t_im = torch.from_numpy(np.asarray(im, np.float32) / 255.0).permute(2,0,1)
        t_m  = (torch.from_numpy(np.asarray(m, np.float32) / 255.0) > 0.5).float()[None]
        return t_im, t_m

dl_inp = DataLoader(InpaintDS(clean_imgs, 512), batch_size=6, shuffle=True, num_workers=2)
migan = CleanMigan512().cuda()
opt = torch.optim.AdamW(migan.parameters(), lr=4e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=15)
for ep in range(15):
    tot = 0.0
    for im, m in dl_inp:
        im, m = im.cuda(), m.cuda()
        pred = migan(im, m)
        hole_l1 = (torch.abs(pred - im) * m).sum() / (m.sum() + 1e-6)
        loss = hole_l1 + 0.08 * lpips_fn(pred * 2 - 1, im * 2 - 1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()
    sched.step()
    if (ep + 1) in (1, 8, 15): print(f"Clean MI-GAN Epoch {ep+1}/15 loss={tot/len(dl_inp):.4f}")

p_mig = '/content/out/migan_clean_512.onnx'
torch.onnx.export(migan.eval().cpu(), (torch.randn(1,3,512,512), torch.ones(1,1,512,512)), p_mig, opset_version=17, dynamo=False,
                  input_names=['image', 'mask'], output_names=['output'])
onnx.checker.check_model(p_mig)
d = open(p_mig, 'rb').read()
shutil.copy2(p_mig, '/content/drive/MyDrive/infinite_ml/migan_clean_512.onnx')
print(f"Clean MI-GAN Exported: {len(d)/1e6:.2f} MB sha256={hashlib.sha256(d).hexdigest()}")
del migan, dl_inp; torch.cuda.empty_cache()


In [ ]:
# ==================== 3/3: #4 PURE-COCO POSE KEYPOINTS (17 COCO keypoints, zero AIC/MPII) ====================
# Trains a lightweight Apache-2.0 MobileNetV4-Small SimCC/Heatmap 17-keypoint head on MS COCO val2017 person crops
import zipfile
coco_ann_zip = '/content/annotations_trainval2017.zip'
if not os.path.exists('/content/annotations/person_keypoints_val2017.json'):
    urllib.request.urlretrieve('http://images.cocodataset.org/annotations/annotations_trainval2017.zip', coco_ann_zip)
    with zipfile.ZipFile(coco_ann_zip) as z:
        z.extract('annotations/person_keypoints_val2017.json', '/content')

coco = json.load(open('/content/annotations/person_keypoints_val2017.json'))
imgs_by_id = {i['id']: i for i in coco['images']}
pose_samples = []
os.makedirs('/content/coco_pose', exist_ok=True)
for an in coco['annotations']:
    if len(pose_samples) >= 400: break
    if an.get('iscrowd') or an.get('num_keypoints', 0) < 6: continue
    im_info = imgs_by_id[an['image_id']]
    x, y, bw, bh = an['bbox']
    if bw < 40 or bh < 60: continue
    lp = f"/content/coco_pose/{im_info['file_name']}"
    if not os.path.exists(lp):
        try: urllib.request.urlretrieve(im_info['coco_url'], lp)
        except Exception: continue
    pose_samples.append((lp, an['bbox'], an['keypoints']))
print('Pure-COCO pose crops ready:', len(pose_samples))

class CleanPoseNet(nn.Module):
    def __init__(self, kpts=17):
        super().__init__()
        self.enc = timm.create_model('mobilenetv4_conv_small.e2400_r224_in1k', pretrained=True, features_only=True, out_indices=(4,))
        c = self.enc.feature_info.channels()[0]
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=False),
            nn.Conv2d(c, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(128, kpts, 1),
        )
    def forward(self, x):
        return self.head(self.enc(x)[0])  # [1, 17, 64, 48] heatmaps for 256x192 input

class PoseDS(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self): return len(self.samples) * 2
    def __getitem__(self, i):
        lp, (bx, by, bw, bh), kpts = self.samples[i % len(self.samples)]
        im = Image.open(lp).convert('RGB')
        W, H = im.size
        cx, cy = bx + bw * 0.5, by + bh * 0.5
        cw, ch = bw * 1.25, bh * 1.25
        x0, y0 = max(0, cx - cw * 0.5), max(0, cy - ch * 0.5)
        x1, y1 = min(W, cx + cw * 0.5), min(H, cy + ch * 0.5)
        crop = im.crop((x0, y0, x1, y1)).resize((192, 256), Image.BILINEAR)
        hm = np.zeros((17, 64, 48), np.float32)
        vis = np.zeros((17, 1, 1), np.float32)
        yy, xx = np.mgrid[0:64, 0:48].astype(np.float32)
        for k in range(17):
            kx, ky, kv = kpts[k*3], kpts[k*3+1], kpts[k*3+2]
            if kv > 0 and x1 > x0 and y1 > y0:
                hx = (kx - x0) / (x1 - x0) * 48.0
                hy = (ky - y0) / (y1 - y0) * 64.0
                if 0 <= hx < 48 and 0 <= hy < 64:
                    hm[k] = np.exp(-((xx - hx)**2 + (yy - hy)**2) / (2 * 2.0**2))
                    vis[k, 0, 0] = 1.0
        return torch.from_numpy(np.asarray(crop, np.float32) / 255.0).permute(2,0,1), torch.from_numpy(hm), torch.from_numpy(vis)

dl_p = DataLoader(PoseDS(pose_samples), batch_size=16, shuffle=True, num_workers=2)
pose_net = CleanPoseNet().cuda()
opt = torch.optim.AdamW(pose_net.parameters(), lr=5e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
for ep in range(20):
    tot = 0.0
    for img, hm, vis in dl_p:
        img, hm, vis = img.cuda(), hm.cuda(), vis.cuda()
        pred = pose_net(img)
        loss = (((pred - hm) ** 2) * vis).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()
    sched.step()
    if (ep + 1) in (1, 10, 20): print(f"Pure-COCO Pose Epoch {ep+1}/20 loss={tot/len(dl_p):.5f}")

p_pose = '/content/out/rtmpose_s_coco_clean.onnx'
torch.onnx.export(pose_net.eval().cpu(), torch.randn(1,3,256,192), p_pose, opset_version=17, dynamo=False, input_names=['input'], output_names=['heatmaps'])
onnx.checker.check_model(p_pose)
d = open(p_pose, 'rb').read()
shutil.copy2(p_pose, '/content/drive/MyDrive/infinite_ml/rtmpose_s_coco_clean.onnx')
print(f"Pure-COCO Pose Exported: {len(d)/1e6:.2f} MB sha256={hashlib.sha256(d).hexdigest()}")
